# ✈️ Orquestador ELT – Flights & Airports

Este notebook ejecuta y documenta el flujo **Extract → Load → Transform (ELT)** del proyecto.

## 🎯 Objetivos
- **Extract:** consultar la API de AviationStack y persistir crudos en **Parquet**.
- **Load:** cargar crudos a **Delta Lake** (control de esquema e histórico).
- **Transform:** aplanar, limpiar y enriquecer datos listos para análisis.

## 🔧 Configuración inicial

In [ ]:
# Asegurar imports y rutas
import os
import pandas as pd
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError
from pathlib import Path
import configparser

# Configuración
script_dir = Path(os.getcwd())
config_path = script_dir.parent / "pipeline.config"

config = configparser.ConfigParser()
config.read(config_path)

# Rutas principales del data lake
DATA_LAKE = script_dir.parent / config.get("paths", "data_lake", fallback="data_lake")
PROCESSED_ENRICHED_PATH = DATA_LAKE / "flights" / "processed_enriched"
AGG_PATH = DATA_LAKE / "flights" / "agg_by_status"


(WindowsPath('../data_lake'),
 WindowsPath('../data_lake/flights/processed'),
 WindowsPath('../data_lake/airports/processed'),
 WindowsPath('../data_lake/flights/processed_enriched_pandas'),
 WindowsPath('../data_lake/flights/agg_by_status'))

## 1️⃣ Extract

Demostración del Flujo Incremental y Full

## Paso 1: Configuración Inicial ⚙️
Abre el archivo pipeline.config y verifica la sección [state]. La fecha de last_flights_run es el punto de partida para el proceso incremental. Este es el estado inicial de tu pipeline.

* [state]
last_flights_run = '2025-01-01'

## Paso 2: Extracción y Carga Full (Aeropuertos) ✈️
Ejecuta el script extract.py.

Este script se conectará a la API y extraerá los datos de vuelos para la fecha actual y los datos de aeropuertos completos (full).

Luego, guardará ambos conjuntos de datos como archivos .parquet en sus respectivos directorios.

### 📝 Aclaración Estrategia de Extracción Mejorada: Resiliencia del Pipeline

El script de extracción (extract.py) ahora implementa una estrategia de triple fallback en la extracción de datos de vuelos:

Intento principal (Incremental): Se realiza una llamada a la API para obtener vuelos de la fecha del último procesamiento guardada en el archivo pipeline.config.

Primer Fallback (API sin filtro): Si el primer intento falla (por un error de la API o por no encontrar datos para esa fecha), el script intenta una segunda llamada sin el filtro de fecha. Esto permite obtener los datos más recientes disponibles, aunque no se pueda garantizar la incrementalidad.

Segundo Fallback (Archivo de prueba): Si ambos intentos de la API fallan, el pipeline recurre a un archivo de prueba local (flights_raw_sample.csv). Este archivo contiene un pequeño conjunto de datos de ejemplo que asegura que el flujo de trabajo de carga y transformación pueda continuar y no se detenga por una falla en la fuente de datos.

Esta estrategia garantiza que el pipeline de datos sea más tolerante a fallos, permitiendo que las etapas posteriores (carga y transformación) puedan ejecutarse para fines de prueba, validación o cuando la disponibilidad de la API sea un problema.

In [2]:
!python ../src/extract.py

[2025-08-24 21:02:47] --- 🌍 Iniciando la etapa de Extracción ---
[2025-08-24 21:02:47] 🔎 Buscando vuelos desde la fecha: 2020-01-01
[2025-08-24 21:02:47] 🔗 Consultando API en el endpoint: flights
[2025-08-24 21:02:48] ❌ Error al consultar la API de vuelos: 403 Client Error: Forbidden for url: http://api.aviationstack.com/v1/flights?access_key=fee950e39aa10e46a6c8086f0cf579a2&limit=100&flight_date=2020-01-01
[2025-08-24 21:02:48] ⚠️ La API de vuelos no devolvió datos. Usando archivo de prueba...
[2025-08-24 21:02:48] ✅ Usando el archivo de prueba: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed\flights_raw_sample.csv
[2025-08-24 21:02:48] ✅ Columnas de vuelos renombradas para estandarizar el esquema.
[2025-08-24 21:02:48] ✅ Guardado DataFrame en c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed\flights_raw.parquet
[2025-08-24 21:02:48] 🔗 Consultando API en el endpoint: airports
[2025-08-24 21:02:48] ✅ Datos de 'airports' extraídos: 1

Verifico funcionalidad correcta de "Extract"

In [3]:
import pandas as pd
from pathlib import Path

# Asume que estás en la carpeta raíz de tu proyecto
flights_path = Path('data_lake/flights/processed/flights_raw.parquet')
airports_path = Path('data_lake/airports/processed/airports_raw.parquet')

if flights_path.exists():
    df_flights = pd.read_parquet(flights_path)
    print(f"Filas en flights_raw.parquet: {len(df_flights)}")
else:
    print("El archivo de vuelos no existe.")

if airports_path.exists():
    df_airports = pd.read_parquet(airports_path)
    print(f"Filas en airports_raw.parquet: {len(df_airports)}")
else:
    print("El archivo de aeropuertos no existe.")

El archivo de vuelos no existe.
El archivo de aeropuertos no existe.


## 2️⃣ Load

Ejecuta el script load.py.

Este script leerá los archivos Parquet.

Para los datos de aeropuertos, utilizará la estrategia full (hard_overwrite=True), borrando la tabla Delta existente y escribiendo el conjunto completo de datos estáticos.

Después de este paso, tu tabla Delta de aeropuertos contendrá todos los datos.

In [4]:
!python ../src/load.py

[2025-08-24 21:02:51] --- 📄 Iniciando la etapa de Carga ---
[2025-08-24 21:02:51] ✅ Archivo cargado desde: c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\data_lake\flights\processed\flights_raw.parquet - Filas: 5
[2025-08-24 21:02:51] ✅ Procesando datos de vuelos...
[2025-08-24 21:02:51] 🔄 Limpiando y preparando el DataFrame para Delta Lake...
Traceback (most recent call last):
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\load.py", line 73, in <module>
    main()
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\load.py", line 48, in main
    save_df_as_delta(
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\utils.py", line 97, in save_df_as_delta
    write_deltalake(
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\venv\Lib\site-packages\deltalake\writer.py", line 316, in write_deltalake
    write_deltalake_rust(
_internal.SchemaMismatchError: Cannot cast schema, number of fields does not match: 5 vs 8


## Extracción y Carga Incremental (Vuelos) 🛫
### Para simular una extracción incremental, debes correr el proceso una segunda vez.

Ejecuta extract.py y load.py nuevamente.

* extract.py: Leerá la fecha actualizada en pipeline.config y buscará nuevos datos desde esa fecha. El script actualizará la fecha en pipeline.config una vez que la extracción de vuelos sea exitosa.

* load.py: Leerá los nuevos datos de vuelos del archivo Parquet y, en lugar de borrar la tabla, los anexará a la tabla Delta de vuelos existente (mode="append").

### Verificación:

Abre pipeline.config y verás que la fecha de last_flights_run se actualizó a la fecha de la ejecución actual, demostrando que el proceso es stateful.

## 3️⃣ Transform

Ejecuta el script transform.py.

* Este script leerá todos los datos de la tabla Delta de vuelos (flights_processed_enriched), incluyendo los de la primera y segunda carga.

* Realizará la limpieza, el aplanamiento y el enriquecimiento de todo el historial acumulado.

In [5]:
!python ../src/transform.py

[2025-08-24 21:02:52] --- 🔄 Iniciando la etapa de Transformación ---
[2025-08-24 21:02:52] ✅ Tabla Delta de vuelos cargada: 4 filas.
[2025-08-24 21:02:52] ✅ Tabla Delta de aeropuertos cargada: 4 filas.
[2025-08-24 21:02:52] 🧹 Limpiando y aplanando DataFrame de vuelos...
[2025-08-24 21:02:52] 🤝 Enriqueciendo datos...
Traceback (most recent call last):
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\transform.py", line 137, in <module>
    main()
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\transform.py", line 103, in main
    df_enriched = clean_and_enrich_data(df_flights_raw, df_airports_raw)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\src\transform.py", line 68, in clean_and_enrich_data
    df_enriched = pd.merge(
                  ^^^^^^^^^
  File "c:\Users\MONSO\OneDrive\Escritorio\Proyecto-ELT\venv\Lib\site-packages\pandas\core\reshape\merge.py", line 170, in merge
    op

## 4️⃣ Validación rápida de resultados 

Leemos las tablas Delta generadas y mostramos conteos básicos para asegurar que el pipeline corrió.


In [8]:
import pandas as pd
from pathlib import Path
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError
import os

# --- Rutas de archivos y tablas ---
DATA_LAKE_PATH = Path("../data_lake")
FLIGHTS_RAW_PARQUET = DATA_LAKE_PATH / "flights" / "processed" / "flights_raw.parquet"
AIRPORTS_RAW_PARQUET = DATA_LAKE_PATH / "airports" / "processed" / "airports_raw.parquet"
FLIGHTS_DELTA_PATH = DATA_LAKE_PATH / "flights" / "delta"
AIRPORTS_DELTA_PATH = DATA_LAKE_PATH / "airports" / "delta"
PROCESSED_ENRICHED_PATH = DATA_LAKE_PATH / "flights" / "processed_enriched"
AGG_PATH = DATA_LAKE_PATH / "flights" / "agg_by_status"

# --- Funciones de ayuda ---
def get_parquet_row_count(path: Path) -> int:
    """Retorna el número de filas de un archivo Parquet si existe, de lo contrario 0."""
    if path.exists():
        try:
            return pd.read_parquet(path).shape[0]
        except Exception:
            return 0
    return 0

def get_delta_table_row_count(path: Path) -> int:
    """Retorna el número de filas de una tabla Delta si existe, de lo contrario 0."""
    try:
        if path.exists():
            return DeltaTable(str(path)).to_pandas().shape[0]
    except TableNotFoundError:
        return 0
    return 0

# --- Lógica principal ---
print("--- 📄 Resumen del Flujo de Datos ---")

# Recopilar métricas de cada etapa
summary = {
    "Etapa": ["Extract", "Extract", "Load", "Load", "Transform", "Transform"],
    "Descripción": [
        "Vuelos crudos (flights_raw.parquet)",
        "Aeropuertos crudos (airports_raw.parquet)",
        "Vuelos en Delta (flights/delta)",
        "Aeropuertos en Delta (airports/delta)",
        "Datos enriquecidos (processed_enriched)",
        "Datos agregados (agg_by_status)"
    ],
    "Recuento de Filas": [
        get_parquet_row_count(FLIGHTS_RAW_PARQUET),
        get_parquet_row_count(AIRPORTS_RAW_PARQUET),
        get_delta_table_row_count(FLIGHTS_DELTA_PATH),
        get_delta_table_row_count(AIRPORTS_DELTA_PATH),
        get_delta_table_row_count(PROCESSED_ENRICHED_PATH),
        get_delta_table_row_count(AGG_PATH)
    ],
    "Estado": ["❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado", "❌ No encontrado"]
}

# Actualizar el estado basado en la existencia del archivo/tabla
for i, count in enumerate(summary["Recuento de Filas"]):
    if count > 0:
        summary["Estado"][i] = "✅ OK"

# Crear y mostrar el DataFrame de resumen
df_summary = pd.DataFrame(summary)

# Reemplazo de df.to_markdown() para evitar la dependencia de 'tabulate'
print(df_summary.to_string(index=False))

print("\n--- ℹ️ Notas adicionales ---")
if not FLIGHTS_RAW_PARQUET.exists() or os.path.getsize(FLIGHTS_RAW_PARQUET) == 0:
    print("⚠️ El archivo de vuelos crudos no se generó o está vacío. El proceso de extracción puede haber fallado.")
if not AIRPORTS_RAW_PARQUET.exists() or os.path.getsize(AIRPORTS_RAW_PARQUET) == 0:
    print("⚠️ El archivo de aeropuertos crudos no se generó o está vacío. El proceso de extracción puede haber fallado.")
if not FLIGHTS_DELTA_PATH.exists() or get_delta_table_row_count(FLIGHTS_DELTA_PATH) == 0:
    print("⚠️ La tabla Delta de vuelos no se creó. La etapa de carga puede haber fallado.")
if not AIRPORTS_DELTA_PATH.exists() or get_delta_table_row_count(AIRPORTS_DELTA_PATH) == 0:
    print("⚠️ La tabla Delta de aeropuertos no se creó. La etapa de carga puede haber fallado.")
if not PROCESSED_ENRICHED_PATH.exists() or get_delta_table_row_count(PROCESSED_ENRICHED_PATH) == 0:
    print("⚠️ La tabla de datos enriquecidos no se creó. La etapa de transformación puede haber fallado.")
if not AGG_PATH.exists() or get_delta_table_row_count(AGG_PATH) == 0:
    print("⚠️ La tabla de datos agregados no se creó. La etapa de transformación de agregación puede haber fallado.")

--- 📄 Resumen del Flujo de Datos ---
    Etapa                               Descripción  Recuento de Filas          Estado
  Extract       Vuelos crudos (flights_raw.parquet)                  5            ✅ OK
  Extract Aeropuertos crudos (airports_raw.parquet)                100            ✅ OK
     Load           Vuelos en Delta (flights/delta)                  4            ✅ OK
     Load     Aeropuertos en Delta (airports/delta)                  4            ✅ OK
Transform   Datos enriquecidos (processed_enriched)                  0 ❌ No encontrado
Transform           Datos agregados (agg_by_status)                  0 ❌ No encontrado

--- ℹ️ Notas adicionales ---
⚠️ La tabla de datos enriquecidos no se creó. La etapa de transformación puede haber fallado.
⚠️ La tabla de datos agregados no se creó. La etapa de transformación de agregación puede haber fallado.


## 5️⃣ Métricas y visualizaciones

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError

# --- Rutas de las tablas Delta ---
DATA_LAKE_PATH = Path("../data_lake")
PROCESSED_ENRICHED_PATH = DATA_LAKE_PATH / "flights" / "processed_enriched"
AGG_PATH = DATA_LAKE_PATH / "flights" / "agg_by_status"

# --- Configuración de gráficos ---
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# --- Funciones de ayuda ---
def get_delta_table_as_df(path: Path) -> pd.DataFrame:
    """Carga una tabla Delta como un DataFrame de Pandas."""
    try:
        if path.exists():
            return DeltaTable(str(path)).to_pandas()
        else:
            print(f"⚠️ Advertencia: La tabla en la ruta {path} no se encuentra.")
            return pd.DataFrame()
    except TableNotFoundError:
        print(f"⚠️ Advertencia: No se pudo cargar la tabla Delta en {path}.")
        return pd.DataFrame()

# --- Cargar datos ---
print("Cargando datos para visualización...")
df_enriched = get_delta_table_as_df(PROCESSED_ENRICHED_PATH)
df_agg = get_delta_table_as_df(AGG_PATH)

# --- Generar gráficos si los DataFrames no están vacíos ---

if not df_agg.empty:
    print("Generando gráfico de barras: Total de vuelos por estado...")
    plt.figure(figsize=(10, 6))
    sns.barplot(x="status", y="total_flights", data=df_agg, palette="viridis")
    plt.title("Total de Vuelos por Estado", fontsize=16)
    plt.xlabel("Estado del Vuelo", fontsize=12)
    plt.ylabel("Total de Vuelos", fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print("Gráfico de barras generado con éxito.")

if not df_agg.empty:
    print("Generando gráfico circular: Proporción de estados de vuelo...")
    plt.figure(figsize=(8, 8))
    plt.pie(
        df_agg["total_flights"],
        labels=df_agg["status"],
        autopct="%1.1f%%",
        startangle=140,
        colors=sns.color_palette("viridis", n_colors=len(df_agg))
    )
    plt.title("Proporción de Estados de Vuelo", fontsize=16)
    plt.show()
    print("Gráfico circular generado con éxito.")

if not df_enriched.empty:
    print("Generando gráfico de líneas: Vuelos diarios...")
    df_flights_by_date = df_enriched.groupby("flight_date").size().reset_index(name="count")
    
    plt.figure(figsize=(12, 6))
    sns.lineplot(x="flight_date", y="count", data=df_flights_by_date, marker="o")
    plt.title("Vuelos Diarios", fontsize=16)
    plt.xlabel("Fecha", fontsize=12)
    plt.ylabel("Número de Vuelos", fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print("Gráfico de líneas generado con éxito.")

if df_agg.empty and df_enriched.empty:
    print("❌ No se pudieron generar gráficos. Asegúrate de que las etapas de carga y transformación se ejecutaron correctamente y que los archivos Delta no están vacíos.")

Cargando datos para visualización...
⚠️ Advertencia: La tabla en la ruta ..\data_lake\flights\processed_enriched no se encuentra.
⚠️ Advertencia: La tabla en la ruta ..\data_lake\flights\agg_by_status no se encuentra.
❌ No se pudieron generar gráficos. Asegúrate de que las etapas de carga y transformación se ejecutaron correctamente y que los archivos Delta no están vacíos.
